In [4]:
import pandas as pd

df = pd.read_parquet("hf://datasets/ButterChicken98/plantvillage-image-text-pairs/data/train-00000-of-00001.parquet")

In [5]:
df

,image,caption,captions
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,[A tomato leaf showing dark brown lesions and ...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,[A tomato leaf with mosaic-like patterns of li...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,"[A fresh green bell pepper leaf with a smooth,..."
...,...,...,...
20633,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Potato Early blight,[A potato leaf with concentric brown rings for...
20634,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Spider mites Two spotted spider mite,[A tomato leaf infested with two-spotted spide...
20635,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Potato Late blight,"[A potato leaf showing dark, water-soaked lesi..."
20636,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Spider mites Two spotted spider mite,[A tomato leaf infested with two-spotted spide...


In [6]:
df = df.drop(columns= ["image"])
df

,caption,captions
0,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
1,Tomato Late blight,[A tomato leaf showing dark brown lesions and ...
2,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
3,Tomato mosaic virus,[A tomato leaf with mosaic-like patterns of li...
4,Pepper bell healthy,"[A fresh green bell pepper leaf with a smooth,..."
...,...,...
20633,Potato Early blight,[A potato leaf with concentric brown rings for...
20634,Tomato Spider mites Two spotted spider mite,[A tomato leaf infested with two-spotted spide...
20635,Potato Late blight,"[A potato leaf showing dark, water-soaked lesi..."
20636,Tomato Spider mites Two spotted spider mite,[A tomato leaf infested with two-spotted spide...


In [7]:
df = df.explode("captions" , ignore_index=True)
df

,caption,captions
0,Tomato healthy,A vibrant green and healthy tomato leaf with s...
1,Tomato healthy,"A healthy Solanum lycopersicum leaf, free of d..."
2,Tomato healthy,"A fresh tomato leaf outdoors, glowing in sunli..."
3,Tomato healthy,"A clean and healthy tomato leaf image, perfect..."
4,Tomato Late blight,A tomato leaf showing dark brown lesions and w...
...,...,...
82547,Tomato Spider mites Two spotted spider mite,An image of a tomato leaf displaying symptoms ...
82548,Tomato Septoria leaf spot,"A tomato leaf showing small, circular brown sp..."
82549,Tomato Septoria leaf spot,A tomato plant leaf infected with Septoria lyc...
82550,Tomato Septoria leaf spot,A tomato leaf outdoors with visible signs of S...


In [8]:
len(df["caption"].unique().tolist())

15

In [9]:
df.to_csv("dataset.csv" , index = False)

In [10]:
df = pd.read_csv("dataset.csv")
df

,caption,captions
0,Tomato healthy,A vibrant green and healthy tomato leaf with s...
1,Tomato healthy,"A healthy Solanum lycopersicum leaf, free of d..."
2,Tomato healthy,"A fresh tomato leaf outdoors, glowing in sunli..."
3,Tomato healthy,"A clean and healthy tomato leaf image, perfect..."
4,Tomato Late blight,A tomato leaf showing dark brown lesions and w...
...,...,...
82547,Tomato Spider mites Two spotted spider mite,An image of a tomato leaf displaying symptoms ...
82548,Tomato Septoria leaf spot,"A tomato leaf showing small, circular brown sp..."
82549,Tomato Septoria leaf spot,A tomato plant leaf infected with Septoria lyc...
82550,Tomato Septoria leaf spot,A tomato leaf outdoors with visible signs of S...


In [11]:
# df = df.drop(columns= ["Unnamed: 0"])

In [12]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [13]:
encoder = LabelEncoder()
encoder.fit(df["caption"].tolist())
df["label"] = encoder.transform(df["caption"].tolist())
df

,caption,captions,label
0,Tomato healthy,A vibrant green and healthy tomato leaf with s...,13
1,Tomato healthy,"A healthy Solanum lycopersicum leaf, free of d...",13
2,Tomato healthy,"A fresh tomato leaf outdoors, glowing in sunli...",13
3,Tomato healthy,"A clean and healthy tomato leaf image, perfect...",13
4,Tomato Late blight,A tomato leaf showing dark brown lesions and w...,7
...,...,...,...
82547,Tomato Spider mites Two spotted spider mite,An image of a tomato leaf displaying symptoms ...,10
82548,Tomato Septoria leaf spot,"A tomato leaf showing small, circular brown sp...",9
82549,Tomato Septoria leaf spot,A tomato plant leaf infected with Septoria lyc...,9
82550,Tomato Septoria leaf spot,A tomato leaf outdoors with visible signs of S...,9


In [14]:
df_train, df_test = train_test_split(df, train_size=0.8)

In [15]:
from datasets import Dataset

In [16]:
train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

In [17]:
!pip install transformers

In [18]:
from transformers import AutoTokenizer

In [19]:
model_name = "distilbert-base-uncased"

In [20]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [21]:
def mapper_function(data):
  return tokenizer(data['captions'], truncation=True)

In [22]:
tokenized_train = train_dataset.map(mapper_function, batched=True)
tokenized_test = test_dataset.map(mapper_function, batched=True)

Map:   0%|          | 0/66041 [00:00<?, ? examples/s]

Map:   0%|          | 0/16511 [00:00<?, ? examples/s]

In [23]:
from transformers import AutoModelForSequenceClassification

In [24]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=15)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00


In [26]:
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
import evaluate
import numpy as np

In [27]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [28]:
eval_metric = evaluate.load("accuracy")

In [29]:
def compute_metrics(pred):
  # Access the true labels from the EvalPrediction object
  labels = pred.label_ids
  logits = pred.predictions
  prediction = np.argmax(logits, axis=1)
  return eval_metric.compute(predictions=prediction, references=labels)

In [30]:
training_arguments = TrainingArguments(output_dir="checkpoints",
                                       per_device_train_batch_size= 16,
                                       per_device_eval_batch_size= 16,
                                       learning_rate= 1e-4,
                                       num_train_epochs=5,
                                       weight_decay=0.01,
                                       logging_strategy="epoch",
                                       save_strategy="epoch",
                                       save_total_limit=2,
                                       report_to="none")


In [31]:
trainer = Trainer(model=model,
                  args=training_arguments,
                  train_dataset=tokenized_train,
                  eval_dataset=tokenized_test,
                  data_collator=data_collator,
                  compute_metrics=compute_metrics,
                  tokenizer= tokenizer)

/tmp/ipython-input-1938101212.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model,


In [32]:
trainer.train()

Step,Training Loss
4128,0.023300
8256,0.000000
12384,0.000000
16512,0.000000
20640,0.000000


TrainOutput(global_step=20640, training_loss=0.004669962751555685, metrics={'train_runtime': 1278.4555, 'train_samples_per_second': 258.284, 'train_steps_per_second': 16.144, 'total_flos': 2411011941054390.0, 'train_loss': 0.004669962751555685, 'epoch': 5.0})

In [45]:
from google.colab import drive
import os
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [46]:
#saving the model

save_dir = "/content/drive/MyDrive/main_model"
os.makedirs(save_dir, exist_ok=True)

final_model_dir = os.path.join(save_dir, "final_model")
os.makedirs(final_model_dir, exist_ok=True)

trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

checkpoints_dir = os.path.join(save_dir, "checkpoints")
os.makedirs(checkpoints_dir, exist_ok=True)

trainer.save_state()
!cp -r checkpoints {checkpoints_dir}

print("Final model saved at: {final_model_dir}")
print("Checkpoints saved at: {checkpoints_dir}")

Final model saved at: {final_model_dir}
Checkpoints saved at: {checkpoints_dir}


In [47]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

final_model_dir= "/content/drive/MyDrive/distilbert_plantvillage_text_model"

loaded_model = AutoModelForSequenceClassification.from_pretrained(final_model_dir)
loaded_tokenizer = AutoTokenizer.from_pretrained(final_model_dir)

trainer = Trainer(
    model=loaded_model,
    args=training_arguments,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=loaded_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,

)

/tmp/ipython-input-3700199964.py:9: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [48]:
predictions_output = trainer.predict(tokenized_test)
preds = np.argmax(predictions_output.predictions, axis=-1)

print("Predicted label indices:", preds[:20])
print("Decoded predictions:", encoder.inverse_transform(preds[:20]))

Predicted label indices: [ 5 11  9 10 11  5  5  0  1  2  1 13 10  8 10  6  8  7 12 12]
Decoded predictions: ['Tomato Bacterial spot' 'Tomato Target Spot' 'Tomato Septoria leaf spot'
 'Tomato Spider mites Two spotted spider mite' 'Tomato Target Spot'
 'Tomato Bacterial spot' 'Tomato Bacterial spot'
 'Pepper bell Bacterial spot' 'Pepper bell healthy' 'Potato Early blight'
 'Pepper bell healthy' 'Tomato healthy'
 'Tomato Spider mites Two spotted spider mite' 'Tomato Leaf Mold'
 'Tomato Spider mites Two spotted spider mite' 'Tomato Early blight'
 'Tomato Leaf Mold' 'Tomato Late blight' 'Tomato YellowLeaf Curl Virus'
 'Tomato YellowLeaf Curl Virus']


In [49]:
true_labels = df_test["label"].tolist()[:20] # Corrected to use .tolist()
decoded_true_labels = encoder.inverse_transform(true_labels)
decoded_true_labels

array(['Tomato Bacterial spot', 'Tomato Target Spot',
       'Tomato Septoria leaf spot',
       'Tomato Spider mites Two spotted spider mite',
       'Tomato Target Spot', 'Tomato Bacterial spot',
       'Tomato Bacterial spot', 'Pepper bell Bacterial spot',
       'Pepper bell healthy', 'Potato Early blight',
       'Pepper bell healthy', 'Tomato healthy',
       'Tomato Spider mites Two spotted spider mite', 'Tomato Leaf Mold',
       'Tomato Spider mites Two spotted spider mite',
       'Tomato Early blight', 'Tomato Leaf Mold', 'Tomato Late blight',
       'Tomato YellowLeaf Curl Virus', 'Tomato YellowLeaf Curl Virus'],
      dtype='<U43')